# 04 — Best Practices and Newer Tools

You now know the fundamentals: `pip`, `venv`, and `.gitignore`. These work — but the workflow has friction that modern tools eliminate.

This module introduces **lockfiles** (the right way to pin dependencies) and three tools that each solve the problem in their own way: `uv`, `poetry`, and `pipenv`.

By the end you'll be able to:
- Explain why `pip freeze` is a workaround, not a real lockfile
- Use `uv` to manage a project end-to-end with a single tool
- Read and understand a `pyproject.toml`
- Choose between `uv`, `poetry`, and `pip + venv` for different situations

---

## 1. The problem with `pip freeze`

`pip freeze` produces a snapshot of your current environment. It's better than nothing, but it has two key limitations:

1. **It captures everything** — including packages you didn't explicitly ask for. Your `requirements.txt` becomes a mix of your real deps and transitive deps, which makes it hard to maintain.

2. **Re-resolving is not deterministic** — if you create a fresh environment and add one new package, pip re-runs its solver. Sub-dependency versions may shift even though you only changed one thing.

In [1]:
import sys
import subprocess

# See what pip freeze produces for you right now
result = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
lines = result.stdout.strip().splitlines()

print(f"pip freeze output: {len(lines)} packages")
print()

# Now imagine you asked for only 2 packages but got back 10+
# The extra lines are transitive deps — pip freeze can't distinguish
print("The problem: which of these did YOU ask for?")
print("(pip freeze can't tell — it just lists everything)")
print()
for line in lines[:10]:
    print(f"  {line}")
if len(lines) > 10:
    print(f"  ... and {len(lines) - 10} more")

pip freeze output: 300 packages

The problem: which of these did YOU ask for?
(pip freeze can't tell — it just lists everything)

  accelerate==1.15.0
  aiofiles==24.1.0
  aiohappyeyeballs==2.7.1
  aiohttp==3.14.3
  aiosignal==1.4.0
  aiosqlite==0.22.1
  annotated-doc==0.0.5
  annotated-types==0.8.0
  anthropic==1.5.0
  anyio==4.14.2
  ... and 290 more


In [2]:
# Distinguish between what you asked for vs. what was pulled in
from importlib.metadata import distributions, requires

def get_all_installed() -> dict[str, str]:
    return {d.metadata["Name"]: d.metadata["Version"] 
            for d in distributions() 
            if d.metadata["Name"]}

def get_transitive_deps(packages: list[str]) -> set[str]:
    """Given top-level packages, find all their transitive deps."""
    visited = set()
    queue = list(packages)
    while queue:
        pkg = queue.pop().lower()
        if pkg in visited:
            continue
        visited.add(pkg)
        try:
            reqs = requires(pkg) or []
            for r in reqs:
                if "; extra ==" not in r:
                    dep_name = r.split()[0].rstrip(";")
                    queue.append(dep_name)
        except Exception:
            pass
    return visited

# Show transitive deps for 'rich'
if "rich" in {d.metadata["Name"] for d in distributions()}:
    deps = get_transitive_deps(["rich"])
    deps.discard("rich")
    print(f"'rich' pulls in these transitive dependencies:")
    for dep in sorted(deps):
        print(f"  {dep}")
    print()
    print("All of these appear in 'pip freeze' — indistinguishable from your own deps.")
else:
    print("'rich' not installed — run: pip install rich")

'rich' pulls in these transitive dependencies:
  markdown-it-py
  mdurl~=0.1
  pygments

All of these appear in 'pip freeze' — indistinguishable from your own deps.


---

## 2. Lockfiles: the right solution

A **lockfile** is a machine-managed file that records the complete, solved dependency graph — every package, every exact version, and a cryptographic hash to verify the download.

The key difference from `pip freeze`:

| | `requirements.txt` (from freeze) | Lockfile (uv.lock, poetry.lock) |
|---|---|---|
| Managed by | You, manually | Tool, automatically |
| Separates your deps from transitive | No | Yes (pyproject.toml = yours, lockfile = full tree) |
| Hash verification | No | Yes |
| Re-resolves on each install | Yes | No — replays the recorded solution |
| Update process | `pip freeze` again | `uv lock --upgrade` / `poetry update` |

In [3]:
# Anatomy of a uv.lock entry

uv_lock_entry = """\
[[package]]
name = "httpx"
version = "0.27.0"
source = { registry = "https://pypi.org/simple" }
dependencies = [
  { name = "anyio" },
  { name = "certifi" },
  { name = "httpcore" },
  { name = "idna" },
  { name = "sniffio" },
]

[package.metadata]
requires-dist = [
  { name = "anyio",     specifier = ">=3.0" },
  { name = "certifi",   specifier = "" },
  { name = "httpcore",  specifier = "==1.*" },
]

[[package.metadata.requires-dist]]
wheels = [
  { url = "https://files.pythonhosted.org/packages/.../httpx-0.27.0-py3-none-any.whl",
    hash = "sha256:71d6ee2cdc92d35a27bc0df5fd0f0e9d2c94aa22d8e48cf6e29e5b03c8b11f45" },
]
"""

print("A lockfile entry contains:")
print("  1. The exact version")
print("  2. All direct dependencies of this package")
print("  3. The download URL + sha256 hash")
print()
print(uv_lock_entry)

A lockfile entry contains:
  1. The exact version
  2. All direct dependencies of this package
  3. The download URL + sha256 hash

[[package]]
name = "httpx"
version = "0.27.0"
source = { registry = "https://pypi.org/simple" }
dependencies = [
  { name = "anyio" },
  { name = "certifi" },
  { name = "httpcore" },
  { name = "idna" },
  { name = "sniffio" },
]

[package.metadata]
requires-dist = [
  { name = "anyio",     specifier = ">=3.0" },
  { name = "certifi",   specifier = "" },
  { name = "httpcore",  specifier = "==1.*" },
]

[[package.metadata.requires-dist]]
wheels = [
  { url = "https://files.pythonhosted.org/packages/.../httpx-0.27.0-py3-none-any.whl",
    hash = "sha256:71d6ee2cdc92d35a27bc0df5fd0f0e9d2c94aa22d8e48cf6e29e5b03c8b11f45" },
]



In [4]:
# What the hash protects against
import hashlib

# Simulate: we downloaded a package wheel file
wheel_content_ok = b"This is the genuine httpx wheel contents..."
wheel_content_tampered = b"This is a malicious replacement of httpx..."

expected_hash = hashlib.sha256(wheel_content_ok).hexdigest()

def verify_download(content: bytes, expected: str) -> bool:
    actual = hashlib.sha256(content).hexdigest()
    return actual == expected

print("Scenario: supply chain attack — someone swaps a package on PyPI")
print()
print(f"Expected hash : sha256:{expected_hash[:32]}...")
print()
print("Genuine download   :", "PASS" if verify_download(wheel_content_ok, expected_hash) else "FAIL")
print("Tampered download  :", "PASS" if verify_download(wheel_content_tampered, expected_hash) else "FAIL (rejected!!)")
print()
print("Lockfiles store the expected hash.")
print("Any tampered download is detected and rejected before installation.")

Scenario: supply chain attack — someone swaps a package on PyPI

Expected hash : sha256:a59c33bf14dafd6ddd60171c379360dd...

Genuine download   : PASS
Tampered download  : FAIL (rejected!!)

Lockfiles store the expected hash.
Any tampered download is detected and rejected before installation.


---

## 3. uv — fast, all-in-one

`uv` is written in Rust by [Astral](https://astral.sh). It replaces `pip`, `venv`, `pip-tools`, and optionally `pyenv` with a single binary.

**Key advantages:**
- **Speed** — 10–100x faster than pip (Rust-based resolver and installer)
- **One tool** — manages venvs, packages, Python versions, and project config
- **Proper lockfile** — `uv.lock` separates your deps from transitive deps
- **Hash verification** — all downloaded packages are checked
- **Growing fast** — FastAPI, Django, and many major projects now recommend it

In [5]:
import subprocess, shutil

uv_path = shutil.which("uv")
if uv_path:
    result = subprocess.run(["uv", "--version"], capture_output=True, text=True)
    print("uv is installed:", result.stdout.strip())
    print()
    print("Try:")
    print("  uv python list              # see available Python versions")
    print("  uv init my-project          # start a new project")
    print("  uv add requests             # add a dependency")
    print("  uv sync                     # install from lockfile")
else:
    print("uv is not installed.")
    print()
    print("Install with:")
    print("  pip install uv")
    print()
    print("Or the standalone installer (recommended — no Python needed):")
    print("  # macOS/Linux:")
    print("  curl -LsSf https://astral.sh/uv/install.sh | sh")
    print("  # Windows:")
    print("  powershell -ExecutionPolicy ByPass -c \"irm https://astral.sh/uv/install.ps1 | iex\"")

uv is not installed.

Install with:
  pip install uv

Or the standalone installer (recommended — no Python needed):
  # macOS/Linux:
  curl -LsSf https://astral.sh/uv/install.sh | sh
  # Windows:
  powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"


### Complete uv workflow

Here is every command you need to know, from project creation to deployment:

In [6]:
uv_workflow = [
    ("Install uv (once)",         "pip install uv"),
    ("Start a project",           "uv init my-project"),
    ("Enter the project",         "cd my-project"),
    ("Add a dependency",          "uv add requests"),
    ("Add a dev dependency",      "uv add --dev pytest ruff"),
    ("Remove a dependency",       "uv remove requests"),
    ("Run code in the venv",      "uv run python app.py"),
    ("Run tests",                 "uv run pytest"),
    ("Upgrade all deps",          "uv lock --upgrade"),
    ("Upgrade one dep",           "uv lock --upgrade-package httpx"),
    ("Sync from lockfile",        "uv sync"),
    ("Sync dev deps too",         "uv sync --dev"),
    ("Show dep tree",             "uv tree"),
    ("Install Python 3.12",       "uv python install 3.12"),
    ("Use Python 3.12 here",      "uv python pin 3.12"),
]

print(f"{'Action':<28} {'Command'}")
print("-" * 70)
for action, cmd in uv_workflow:
    print(f"{action:<28} {cmd}")

Action                       Command
----------------------------------------------------------------------
Install uv (once)            pip install uv
Start a project              uv init my-project
Enter the project            cd my-project
Add a dependency             uv add requests
Add a dev dependency         uv add --dev pytest ruff
Remove a dependency          uv remove requests
Run code in the venv         uv run python app.py
Run tests                    uv run pytest
Upgrade all deps             uv lock --upgrade
Upgrade one dep              uv lock --upgrade-package httpx
Sync from lockfile           uv sync
Sync dev deps too            uv sync --dev
Show dep tree                uv tree
Install Python 3.12          uv python install 3.12
Use Python 3.12 here         uv python pin 3.12


In [7]:
# What 'uv init my-project' creates

uv_project_structure = """\
my-project/
├── .venv/                  ← created automatically by uv
├── .python-version         ← pinned Python version (e.g. '3.11')
├── pyproject.toml          ← project metadata and dependencies
├── uv.lock                 ← lockfile (commit this)
├── README.md
└── hello.py                ← starter script
"""

print("uv init creates this structure:")
print(uv_project_structure)

starter_pyproject = """\
[project]
name = "my-project"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.11"
dependencies = []
"""

print("Initial pyproject.toml:")
print(starter_pyproject)

uv init creates this structure:
my-project/
├── .venv/                  ← created automatically by uv
├── .python-version         ← pinned Python version (e.g. '3.11')
├── pyproject.toml          ← project metadata and dependencies
├── uv.lock                 ← lockfile (commit this)
├── README.md
└── hello.py                ← starter script

Initial pyproject.toml:
[project]
name = "my-project"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.11"
dependencies = []



---

## 4. poetry — for library authors

`poetry` has been the go-to tool for Python library authors since 2018. It's mature, widely used, and particularly strong when you need to **build and publish** a package to PyPI.

In [8]:
poetry_workflow = [
    ("Install poetry",           "pip install poetry"),
    ("Start a new library",      "poetry new my-lib"),
    ("Start from existing dir",  "poetry init"),
    ("Add a dependency",         "poetry add httpx"),
    ("Add a dev dependency",     "poetry add --group dev pytest"),
    ("Remove a dependency",      "poetry remove httpx"),
    ("Activate the env",         "poetry shell"),
    ("Run without activating",   "poetry run python main.py"),
    ("Install from lockfile",    "poetry install"),
    ("Upgrade all",              "poetry update"),
    ("Check for outdated",       "poetry show --outdated"),
    ("Build for PyPI",           "poetry build"),
    ("Publish to PyPI",          "poetry publish"),
]

print(f"{'Action':<28} {'Command'}")
print("-" * 70)
for action, cmd in poetry_workflow:
    print(f"{action:<28} {cmd}")

Action                       Command
----------------------------------------------------------------------
Install poetry               pip install poetry
Start a new library          poetry new my-lib
Start from existing dir      poetry init
Add a dependency             poetry add httpx
Add a dev dependency         poetry add --group dev pytest
Remove a dependency          poetry remove httpx
Activate the env             poetry shell
Run without activating       poetry run python main.py
Install from lockfile        poetry install
Upgrade all                  poetry update
Check for outdated           poetry show --outdated
Build for PyPI               poetry build
Publish to PyPI              poetry publish


In [9]:
# A pyproject.toml as poetry creates it

poetry_pyproject = """\
[tool.poetry]
name = "my-http-lib"
version = "0.2.0"
description = "A thin wrapper around httpx"
authors = ["Your Name <you@example.com>"]
license = "MIT"
readme = "README.md"

[tool.poetry.dependencies]
python = "^3.11"
httpx = ">=0.25,<2.0"   # required at runtime
pydantic = "^2.0"       # ^2.0 means >=2.0,<3.0

[tool.poetry.group.dev.dependencies]
pytest = "^8.0"
ruff = "^0.5"
mypy = "^1.10"

[build-system]
requires = ["poetry-core"]
build-backend = "poetry.core.masonry.api"
"""

print("pyproject.toml with poetry:")
print(poetry_pyproject)

pyproject.toml with poetry:
[tool.poetry]
name = "my-http-lib"
version = "0.2.0"
description = "A thin wrapper around httpx"
authors = ["Your Name <you@example.com>"]
license = "MIT"
readme = "README.md"

[tool.poetry.dependencies]
python = "^3.11"
httpx = ">=0.25,<2.0"   # required at runtime
pydantic = "^2.0"       # ^2.0 means >=2.0,<3.0

[tool.poetry.group.dev.dependencies]
pytest = "^8.0"
ruff = "^0.5"
mypy = "^1.10"

[build-system]
requires = ["poetry-core"]
build-backend = "poetry.core.masonry.api"



---

## 5. Anatomy of `pyproject.toml`

`pyproject.toml` is the modern standard for Python project metadata, defined in [PEP 518](https://peps.python.org/pep-0518/) and [PEP 621](https://peps.python.org/pep-0621/). Both `uv` and `poetry` use it, though with slightly different sections.

In [10]:
# A well-commented pyproject.toml for a real application

pyproject_app = """\
[project]
name = "task-tracker"                # package name on PyPI
version = "2.1.0"
description = "A CLI task tracking tool"
requires-python = ">=3.11"           # minimum Python version

# Runtime dependencies — installed when users run `pip install task-tracker`
dependencies = [
    "typer>=0.12",         # CLI framework
    "rich>=13.0",          # terminal formatting
    "sqlmodel>=0.0.18",    # ORM for SQLite storage
    "python-dotenv>=1.0",  # .env loading
]

[project.optional-dependencies]
# Dev tools — installed with: pip install task-tracker[dev]
dev = [
    "pytest>=8.0",
    "ruff>=0.5",
    "mypy>=1.10",
]

[project.scripts]
# After install, `task` becomes a shell command
task = "task_tracker.cli:app"

[tool.ruff]
line-length = 100
target-version = "py311"
select = ["E", "F", "I"]            # pyflakes + isort

[tool.mypy]
strict = true
python_version = "3.11"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"
"""

print(pyproject_app)

[project]
name = "task-tracker"                # package name on PyPI
version = "2.1.0"
description = "A CLI task tracking tool"
requires-python = ">=3.11"           # minimum Python version

# Runtime dependencies — installed when users run `pip install task-tracker`
dependencies = [
    "typer>=0.12",         # CLI framework
    "rich>=13.0",          # terminal formatting
    "sqlmodel>=0.0.18",    # ORM for SQLite storage
    "python-dotenv>=1.0",  # .env loading
]

[project.optional-dependencies]
# Dev tools — installed with: pip install task-tracker[dev]
dev = [
    "pytest>=8.0",
    "ruff>=0.5",
    "mypy>=1.10",
]

[project.scripts]
# After install, `task` becomes a shell command
task = "task_tracker.cli:app"

[tool.ruff]
line-length = 100
target-version = "py311"
select = ["E", "F", "I"]            # pyflakes + isort

[tool.mypy]
strict = true
python_version = "3.11"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"



In [11]:
# Python 3.11+ ships with tomllib to read TOML files
try:
    import tomllib
except ImportError:
    import tomli as tomllib  # pip install tomli for older Python

import io

# Parse the pyproject.toml string we defined above
data = tomllib.loads(pyproject_app)

project = data["project"]
print(f"Project: {project['name']} v{project['version']}")
print(f"Requires Python: {project['requires-python']}")
print()
print("Runtime dependencies:")
for dep in project["dependencies"]:
    print(f"  {dep}")
print()
print("Dev dependencies:")
for dep in project["optional-dependencies"]["dev"]:
    print(f"  {dep}")

Project: task-tracker v2.1.0
Requires Python: >=3.11

Runtime dependencies:
  typer>=0.12
  rich>=13.0
  sqlmodel>=0.0.18
  python-dotenv>=1.0

Dev dependencies:
  pytest>=8.0
  ruff>=0.5
  mypy>=1.10


---

## 6. Tool comparison

Which tool should you use? Here's a practical breakdown:

In [12]:
comparison = [
    ("Feature",            "pip+venv",   "uv",          "poetry",     "pipenv"),
    ("-" * 18,             "-" * 11,      "-" * 11,       "-" * 11,     "-" * 11),
    ("Speed",              "moderate",   "very fast",   "moderate",   "moderate"),
    ("Manages venv",       "manual",     "automatic",   "automatic",  "automatic"),
    ("Lockfile",           "no",         "uv.lock",     "poetry.lock","Pipfile.lock"),
    ("Hash verification",  "no",         "yes",         "yes",        "yes"),
    ("Build & publish",    "no",         "yes",         "yes",        "no"),
    ("Python version mgmt","no",         "yes",         "via plugin", "no"),
    ("Config file",        "requirements","pyproject",  "pyproject",  "Pipfile"),
    ("Adoption trend",     "baseline",   "growing fast","established","declining"),
]

col_widths = [20, 12, 12, 12, 12]
for row in comparison:
    print("".join(str(cell).ljust(w) for cell, w in zip(row, col_widths)))

Feature             pip+venv    uv          poetry      pipenv      
------------------  ----------- ----------- ----------- ----------- 
Speed               moderate    very fast   moderate    moderate    
Manages venv        manual      automatic   automatic   automatic   
Lockfile            no          uv.lock     poetry.lock Pipfile.lock
Hash verification   no          yes         yes         yes         
Build & publish     no          yes         yes         no          
Python version mgmt no          yes         via plugin  no          
Config file         requirementspyproject   pyproject   Pipfile     
Adoption trend      baseline    growing fastestablished declining   


### Decision guide

```
Starting a new project today?
    └── Use uv  (fastest, all-in-one, growing ecosystem)

Building a library you'll publish to PyPI?
    ├── uv       (newer option, full-featured)
    └── poetry   (established, great publishing UX)

Joining an existing project?
    └── Use whatever the project already uses

Restricted environment (no third-party tools allowed)?
    └── pip + venv  (in the standard library)
```

In [13]:
import subprocess, shutil

tools = [
    ("pip",    "pip",    "Standard package manager"),
    ("uv",     "uv",     "Fast all-in-one tool (recommended)"),
    ("poetry", "poetry", "Mature library/publish tool"),
    ("pipenv", "pipenv", "Older combined tool (declining)"),
]

print(f"{'Tool':<10} {'Version':<25} {'Notes'}")
print("-" * 65)
for name, cmd, notes in tools:
    path = shutil.which(cmd)
    if path:
        result = subprocess.run([cmd, "--version"], capture_output=True, text=True)
        version = (result.stdout or result.stderr).strip().split("\n")[0]
        print(f"{name:<10} {version:<25} {notes}")
    else:
        print(f"{name:<10} {'not installed':<25} {notes}")

Tool       Version                   Notes
-----------------------------------------------------------------
pip        not installed             Standard package manager
uv         not installed             Fast all-in-one tool (recommended)
poetry     not installed             Mature library/publish tool
pipenv     not installed             Older combined tool (declining)


---

## 7. Best practices checklist

Use this as a checklist for every new Python project:

In [14]:
checklist = [
    ("Project setup",    [
        "Create a virtual environment (.venv) before installing anything",
        "Use uv or poetry instead of plain pip+venv for new projects",
        "Define dependencies in pyproject.toml, not requirements.txt, for new projects",
    ]),
    ("Dependencies",     [
        "Pin exact versions for applications (==)",
        "Use version ranges for libraries (>=, ~=)",
        "Commit the lockfile (uv.lock or poetry.lock)",
        "Split dev deps from runtime deps",
    ]),
    ("Version control",  [
        ".gitignore includes .venv/, __pycache__/, .env, dist/",
        "Never commit .env — commit .env.example instead",
        "Commit pyproject.toml and lockfile",
    ]),
    ("Security",         [
        "Use a lockfile with hash verification",
        "Regularly run 'pip list --outdated' or 'uv tree --outdated'",
        "Rotate any secret that was accidentally committed",
    ]),
]

for category, items in checklist:
    print(f"\n{category}:")
    for item in items:
        print(f"  ☐  {item}")


Project setup:
  ☐  Create a virtual environment (.venv) before installing anything
  ☐  Use uv or poetry instead of plain pip+venv for new projects
  ☐  Define dependencies in pyproject.toml, not requirements.txt, for new projects

Dependencies:
  ☐  Pin exact versions for applications (==)
  ☐  Use version ranges for libraries (>=, ~=)
  ☐  Commit the lockfile (uv.lock or poetry.lock)
  ☐  Split dev deps from runtime deps

Version control:
  ☐  .gitignore includes .venv/, __pycache__/, .env, dist/
  ☐  Never commit .env — commit .env.example instead
  ☐  Commit pyproject.toml and lockfile

Security:
  ☐  Use a lockfile with hash verification
  ☐  Regularly run 'pip list --outdated' or 'uv tree --outdated'
  ☐  Rotate any secret that was accidentally committed


---

## Summary

| Concept | Key point |
|---|---|
| `pip freeze` | Captures a snapshot but doesn't separate your deps from transitive ones |
| Lockfile | Machine-managed, full dep tree + hashes — reproducible and secure |
| `uv` | Fastest modern tool — venvs + packages + lockfile + Python versions |
| `poetry` | Established tool — best for building and publishing libraries |
| `pyproject.toml` | Modern standard for project config — used by both `uv` and `poetry` |

**The two non-negotiables:**
1. Always use a virtual environment — one per project.
2. Use a tool with a lockfile so your builds are reproducible and verified.

---

## Exercises

1. **Try uv end-to-end** — install `uv`, run `uv init scratch-project`, add `rich` and `httpx`. Open the resulting `uv.lock` file and find:
   - How many packages are locked in total
   - The sha256 hash for the `rich` wheel
   - Which packages `httpx` depends on

2. **Read a real lockfile** — find any open-source Python project on GitHub that uses `uv` or `poetry`. Open the lock file and trace one transitive dependency back to the top-level package that requires it.

3. **Write a `pyproject.toml`** — create a `pyproject.toml` for a fictional CLI tool called `devstats` that:
   - Requires Python 3.11+
   - Uses `typer>=0.12`, `rich>=13`, and `httpx>=0.27` at runtime
   - Has a `dev` group with `pytest>=8`, `ruff>=0.5`, and `mypy>=1.10`
   - Exposes a command `devstats` that calls `devstats.cli:app`
   - Parse it with `tomllib` to verify it's valid TOML

4. **Hash verification simulation** — write a function that takes a file path and an expected sha256 hash and returns `True` if the file matches, `False` if it has been tampered with. Test it with a file you modify between checks.

5. **Migration exercise** — take an existing project that uses `pip + venv + requirements.txt` (or create a simple one). Migrate it to `uv`: initialize with `uv init`, add its dependencies with `uv add`, and confirm the app still runs with `uv run python app.py`.